In [ ]:
import os
import ast
import sys
import json
import warnings
import numpy as np
import pandas as pd
import proplot as pplt
from IPython.display import display,HTML
sys.path.insert(0,'..')
warnings.filterwarnings('ignore')
pplt.rc.update({
    'tick.minor':False,
    'savefig.dpi':300,
    'font.size':9,
    'label.size':9,
    'tick.labelsize':9,
    'legend.fontsize':9})

In [ ]:
with open('../scripts/configs.json','r',encoding='utf-8') as f:
    CONFIGS = json.load(f)
MODELSDIR = CONFIGS['filepaths']['models']
SRCONFIG  = CONFIGS['experiments']['sr']
SEEDS     = SRCONFIG['seeds']

# Manually select the best complexity for each seed after inspecting the equation table below.
# Keys are seeds, values are complexity node counts.
BEST = {42: 11, 72: 11, 102: 11}

In [ ]:
_VAR = {
    'rh':         r'\widehat{\mathrm{RH}}',
    'thetae':     r'\widehat{\theta}_{e}',
    'thetaestar': r'\widehat{\theta}_{e}^{*}',
    'lf':         r'\mathrm{LF}',
    'shf':        r'\mathrm{SHF}',
    'lhf':        r'\mathrm{LHF}',
}

def _fmt(v):
    v = float(v)
    return str(int(v)) if v==int(v) else f'{v:.4g}'

def _render(node):
    if isinstance(node,ast.Name):
        return _VAR.get(node.id,r'\mathrm{'+node.id+'}')
    if isinstance(node,ast.Constant):
        return _fmt(node.value)
    if isinstance(node,ast.UnaryOp) and isinstance(node.op,ast.USub):
        inner = _render(node.operand)
        return f'-{inner}' if isinstance(node.operand,ast.Constant) else f'-({inner})'
    if isinstance(node,ast.BinOp):
        l,r,op = _render(node.left),_render(node.right),node.op
        if isinstance(op,ast.Add):  return f'{l} + {r}'
        if isinstance(op,ast.Sub):  return f'{l} - {r}'
        if isinstance(op,ast.Mult):
            lp = f'({l})' if isinstance(node.left,ast.BinOp) else l
            rp = f'({r})' if isinstance(node.right,ast.BinOp) else r
            return fr'{lp} \times {rp}'
        if isinstance(op,ast.Div):  return fr'\frac{{{l}}}{{{r}}}'
        if isinstance(op,ast.Pow):  return f'{l}^{{{r}}}'
    if isinstance(node,ast.Call):
        fname = node.func.id if isinstance(node.func,ast.Name) else '?'
        a     = _render(node.args[0]) if node.args else ''
        anode = node.args[0] if node.args else None
        cplx  = isinstance(anode,(ast.BinOp,ast.Call,ast.UnaryOp))
        w     = f'({a})' if cplx else a
        if fname=='cube':   return f'{w}^3'
        if fname=='square': return f'{w}^2'
        if fname=='sqrt':   return fr'\sqrt{{{a}}}'
        if fname=='abs':    return f'|{a}|'
        if fname=='neg':    return f'-({a})' if cplx else f'-{a}'
        if fname in ('exp','log','sin','cos'): return fr'\{fname}\!({a})'
        return fr'\mathrm{{{fname}}}({a})'
    return str(ast.unparse(node))

def prettify(eq):
    try:
        return '$' + _render(ast.parse(eq,mode='eval').body) + '$'
    except Exception:
        return eq

def load_equations(runname):
    seedframes = {}
    for seed in SEEDS:
        filepath = os.path.join(MODELSDIR,'sr',f'{runname}_{seed}_equations.csv')
        if not os.path.exists(filepath):
            print(f'Missing: {filepath}')
            continue
        df = pd.read_csv(filepath)
        df['seed'] = seed
        seedframes[seed] = df
    return seedframes

def equation_table(runname):
    seedframes = load_equations(runname)
    if not seedframes:
        return pd.DataFrame()
    rows = []
    for seed,df in seedframes.items():
        for _,row in df.iterrows():
            rows.append({'Seed':seed,'Complexity':int(row['complexity']),
                         'Loss':float(row['loss']),'Equation':prettify(str(row['equation']))})
    return pd.DataFrame(rows).sort_values(['Seed','Complexity']).reset_index(drop=True)

In [ ]:
for runname in SRCONFIG['runs']:
    tbl = equation_table(runname)
    print(f'\n=== {runname} ===')
    display(HTML(tbl.to_html(escape=False,index=False)))

In [ ]:
colors = ['#5BA7DA','#F2C85E','#D42028']
for runname in SRCONFIG['runs']:
    seedframes = load_equations(runname)
    fig,ax = pplt.subplots(refwidth=5,refheight=2)
    ax.format(xlabel='Equation Complexity',xlim=(0,26),ylabel='Training MSE',grid=True,title=runname)
    for i,(seed,df) in enumerate(sorted(seedframes.items())):
        df      = df.sort_values('complexity')
        color   = colors[i%len(colors)]
        ax.plot(df['complexity'],df['loss'],color=color,alpha=0.5,linewidth=1,linestyle='--',zorder=1,label='')
        ax.scatter(df['complexity'],df['loss'],color=color,marker='.',zorder=3,label=f'Seed {seed}')
        best_c = BEST.get(seed)
        if best_c is not None:
            row = df[df['complexity']==best_c]
            if not row.empty:
                ax.scatter([row['complexity'].values[0]],[row['loss'].values[0]],
                           color=color,marker='*',markersize=150,zorder=5,
                           edgecolors='k',linewidths=0.5)
    ax.legend(loc='ur',ncols=1)
    pplt.show()
    # fig.save('../figs/fig_S2.jpg')